In [ ]:
import os
import sys
import yaml
import random
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime

from hydra import initialize, compose
from omegaconf import DictConfig, OmegaConf
from copy import deepcopy


os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"*** Device set to: {device} ***")

# --- PATH CONFIGURATION ---
# 1. Get the directory of the current notebook
notebook_dir = os.getcwd()

# 2. Go up one level to the common parent folder
parent_dir = os.path.dirname(notebook_dir)

# 3. Define the path to the 'train_and_shap' folder
scripts_dir = os.path.join(parent_dir, 'train_and_shap')

# 4. Add paths to system so Python can find your custom modules
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
if scripts_dir not in sys.path:
    sys.path.append(scripts_dir)
print(f"Added to sys.path:\n - {parent_dir}\n - {scripts_dir}")

# 5. Import your custom modules
try:
    from quantifydrivers.train_and_shap.config_schema import validate_schema
    from dataloading_script import build_datasets_and_loaders
    from training_script import training
    from evaluation_script import evaluation
    from SHAP_script import compute_SHAP
    print("\nSUCCESS: Custom modules imported.")
except ImportError as e:
    print(f"\nCRITICAL ERROR: {e}")
    print("Double check that 'train_and_shap' folder contains the scripts (dataloading_script.py, etc).")

# --- HYDRA CONFIGURATION ---
rel_config_path = "../train_and_shap/conf"

try:
    from hydra.core.global_hydra import GlobalHydra
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()

    with initialize(version_base=None, config_path=rel_config_path):
        cfg = compose(config_name="config", overrides=[])

    print("Config Loaded Successfully")
    print(cfg)

except Exception as e:
    print(f"Hydra Error: {e}")

# Validate Config
try:
    validated_cfg = validate_schema(cfg)
    print("Config Validation Passed!")
    print(OmegaConf.to_yaml(validated_cfg))
except Exception as e:
    print("Config Validation Failed or validate_schema not imported.")

In [ ]:
# --- DETERMINISM & LOGGING ---
timestamp = datetime.now()
formatted_time = timestamp.strftime('%m-%d-%Y_%H-%M')
print(f"Timestamp: {formatted_time}")

try:
    torch.use_deterministic_algorithms(True)
    print("Using deterministic algorithms.")
except Exception as e:
    print(f"Could not enforce deterministic algorithms: {e}")

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

# Create Generator and Seed everything
g = torch.Generator()
g.manual_seed(validated_cfg.seed)
random.seed(validated_cfg.seed)
np.random.seed(validated_cfg.seed)
torch.manual_seed(validated_cfg.seed)

print(f"Seeding complete. Seed: {validated_cfg.seed}")

In [ ]:
sites = ['cordoba','lyon','hannover','stockholm','belgrado']

for s,site in enumerate(sites):

    print(f"doing site: {site}")

    # 1. Build Datasets
    print("--- Building Datasets ---")

    config = deepcopy(validated_cfg)
    config.site = site
    datasets = build_datasets_and_loaders(configuration=config, generator=g)

    combined_train_dataset = datasets['combined_train']
    combined_test_dataset = datasets['combined_test']